<a href="https://colab.research.google.com/github/DiasFrazerGroup/Intro-to-probabilistic-modelling-2026/blob/main/Session3a_BOED_WithAnswers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Optimal Experimental Design
*Session 3a — An Introduction to Probabilistic Model Building*

---

## What this notebook covers

You've built probabilistic models, fitted them to data, and seen how latent variables let you represent hidden structure.  Now the question becomes: **How can these concepts be used in experiment design?**

One rather beautiful example is **Bayesian Optimal Experimental Design (BOED)**.  The key insight is that a model doesn't just describe what you know — it also tells you what you *don't* know, and therefore where collecting data would be most valuable.

We'll explore this through a concrete, runnable example: estimating a person's working-memory capacity from a digit-span task.  The model is simple (one latent parameter, binary outcomes), but the ideas generalise directly to drug discovery, protein design, clinical trials, and active learning.

---

## The working memory experiment

The **digit-span task** is a classic cognitive psychology experiment:

- A participant sees a sequence of $l$ digits and tries to recall it.
- The probability of success falls as $l$ increases.
- Each person has a **capacity threshold $\theta$**: the sequence length at which they have a
  50/50 chance of succeeding.

Our Bayesian model:

| Symbol | Role |
|--------|------|
| $l$ | Design variable — sequence length chosen by the experimenter |
| $\theta$ | Latent variable — participant's working-memory capacity |
| $y$ | Observed outcome — 1 (success) or 0 (failure) |

$$
\theta \sim \mathcal{N}(7, 3^2) \qquad \text{(prior: Miller's "7 ± 2" rule of thumb)}
$$
$$
y \sim \text{Bernoulli}\!\left(\sigma\!\left(\theta - l\right)\right) \qquad \text{(logistic success probability)}
$$

---

## Where this fits in the course

| Direction | Session | Concept | How it connects |
|---|---|---|---|
| ← Session 1 | Foundations | Prior / posterior and updating belief | Now we will do this repeatedly |
| ← Session 1 | Foundations | Posterior predictive distribution | Predictive variance over $y$ is the acquisition function that scores candidate experiments |
| → Session 3b | Compressed sensing | Batch vs adaptive design | Compressed sensing = upfront batch design; BOED = sequential adaptive design |

**The key idea:** a probabilistic model quantifies uncertainty.  
BOED is simply a policy that reduces uncertainty as fast as possible by choosing experiments where the posterior predictive variance is highest — i.e., where the model is most surprised by
the outcome either way.

---

## Notebook structure

| Section | Content |
|---------|---------|
| §1 Prior predictive check | What does our model predict *before* seeing data? |
| §2 Inference on mock data | Updating beliefs from a few observations |
| §3 Adaptive experiment (BOED) | Choosing the most informative sequence length each round |
| §4 Naive vs adaptive | Does optimal design actually help? |
| §5 Test yourself | Run the adaptive experiment on your own working memory |

In [ ]:
# Install dependencies (Colab only — comment out if running locally with these already installed)
!pip install pymc arviz scipy matplotlib numpy --quiet

In [ ]:
import logging
logging.getLogger('pymc').setLevel(logging.ERROR)
logging.getLogger('pytensor').setLevel(logging.ERROR)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.stats import norm

import pymc as pm
import arviz as az

print("PyMC:", pm.__version__)
print("ArviZ:", az.__version__)

## §1  Prior predictive check

Before seeing any data, what does our model predict?

We draw several values of $\theta$ from the prior $\mathcal{N}(7, 3^2)$ and plot the
resulting success-probability curves.  This is a **prior predictive check**: it verifies that
the prior encodes sensible beliefs (people with capacity ~7 ± a few digits), rather than
implying impossible behaviour.

The logistic function $\sigma(\theta - l)$ gives the probability of success:
- When $l \ll \theta$ (easy task): probability ≈ 1
- When $l \gg \theta$ (hard task): probability ≈ 0
- When $l = \theta$: probability = 0.5 (the threshold by definition)

In [ ]:
rng = np.random.default_rng(0)

# Prior and model hyperparameters
sensitivity = 1.0   # slope of the logistic curve (steepness of the transition)
prior_mean  = 7.0   # "seven plus or minus two" (Miller 1956)
prior_sd    = 3.0   # allows capacities from ~1 to ~13 within 2 SD

# Draw five hypothetical participants from the prior
theta_samples = rng.normal(prior_mean, prior_sd, size=(5, 1))
print("θ samples from prior N(7, 9):", np.round(theta_samples.ravel(), 2))

# Candidate sequence lengths we might ever test
l_grid = np.arange(1.0, 16.0)

# Logistic success probability for each (participant, length) pair:
#   P(success | θ, l) = σ(θ − l) = 1 / (1 + exp(−(θ − l)))
prob = 1.0 / (1.0 + np.exp(-sensitivity * (theta_samples - l_grid[np.newaxis, :])))

# Plot: each line is one sampled participant
fig, ax = plt.subplots(figsize=(9, 5))
for i, curve in enumerate(prob):
    ax.plot(l_grid, curve, marker="o", label=f"Person {i+1} (θ={theta_samples[i,0]:.1f})")
ax.axhline(0.5, color="gray", ls="--", lw=1, alpha=0.5, label="50% chance threshold")
ax.set_xlabel("Sequence length $l$")
ax.set_ylabel("P(correct recall)")
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_title("Prior predictive check: success curves for five sampled participants")
plt.tight_layout()
plt.show()

## §2  Inference on mock data

Suppose we've already tested a participant at $l = 5, 7, 9$:
- Correct at 5 ✓
- Correct at 7 ✓
- Incorrect at 9 ✗

With just these three trials we can update our posterior over $\theta$.
Even a tiny dataset shifts our beliefs — the posterior will be narrower than the prior
and centred near the boundary between success and failure (around $l = 8$).

In [ ]:
# Observed data from the three trials
l_data = np.array([5.0, 7.0, 9.0])
y_data = np.array([1,   1,   0])     # 1 = correct recall, 0 = incorrect

with pm.Model() as wm_model_demo:
    # Prior: working-memory capacity ~ N(7, 9)
    theta = pm.Normal("theta", mu=prior_mean, sigma=prior_sd)

    # Likelihood: logistic probability of correct recall at each tested length
    # P(y=1 | θ, l) = σ(θ − l)
    logit_p = sensitivity * (theta - l_data)
    p = pm.math.sigmoid(logit_p)
    y = pm.Bernoulli("y", p=p, observed=y_data)

    # MCMC inference
    idata_demo = pm.sample(
        draws=2000, tune=1000, chains=2,
        target_accept=0.9, random_seed=42
    )

az.summary(idata_demo, var_names=["theta"], round_to=2)

The posterior should be **narrower** than the prior and shifted toward
the boundary between success (l=7) and failure (l=9).

The key insight: even 3 binary observations meaningfully constrain the latent capacity.
How can we make each future observation as informative as possible?

In [ ]:
# Extract posterior samples from MCMC
posterior_samples = idata_demo.posterior["theta"].values.flatten()
posterior_mean    = posterior_samples.mean()
posterior_sd      = posterior_samples.std(ddof=1)

# Plot prior and posterior side by side on the same axis
x = np.linspace(0, 14, 300)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, norm.pdf(x, prior_mean, prior_sd),       "r--", lw=2,
        label=f"Prior  N({prior_mean}, {prior_sd**2:.0f})")
ax.plot(x, norm.pdf(x, posterior_mean, posterior_sd), "b-",  lw=2,
        label=f"Posterior ≈ N({posterior_mean:.2f}, {posterior_sd**2:.2f})")

# Mark the observed trials
for l_val, y_val in zip(l_data, y_data):
    ax.axvline(l_val, color="green" if y_val else "red",
               ls=":", lw=1.5, alpha=0.6,
               label=f"l={l_val:.0f} {'✓' if y_val else '✗'}")

ax.set_xlabel(r"$	heta$ (working-memory capacity)")
ax.set_ylabel("Probability density")
ax.set_title("Prior vs posterior after three trials")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Prior SD:     {prior_sd:.2f}")
print(f"Posterior SD: {posterior_sd:.2f}  (uncertainty reduced by "
      f"{(1 - posterior_sd/prior_sd)*100:.0f}%)")

## §3  Bayesian Optimal Experimental Design (BOED)

The experimenter above chose $l = 5, 7, 9$ arbitrarily.  
**BOED asks: which $l$ should we choose to learn most efficiently?**

---

### The information gain criterion

The ideal criterion is the **Expected Information Gain (EIG)**:

$$\text{EIG}(l) = \mathbb{E}_{y \sim p(y|l)}\!\Big[D_{\mathrm{KL}}\big(p(\theta|y,l)\;\|\;p(\theta)\big)\Big]$$

This is hard to compute exactly, but there's a simple, intuitive proxy:

> **Test where the model is most uncertain about the outcome.**

If we're already confident the participant will succeed ($p \approx 1$) or fail ($p \approx 0$),
the trial is not very informative — we'll just confirm what we already know.  
If the model predicts roughly 50/50 ($p \approx 0.5$), the outcome is maximally surprising either way.

Formally, we score each candidate $l$ by the **posterior predictive variance** of $y$:
$$\text{score}(l) = \mathbb{E}_{\theta \sim p(\theta|\text{data})}\!\big[\sigma(\theta-l)\cdot(1-\sigma(\theta-l))\big] + \mathrm{Var}_{\theta}\!\big[\sigma(\theta-l)\big]$$

The first term is the expected Bernoulli variance; the second captures uncertainty about $\theta$ itself.

---

### Adaptive loop

1. Start with the prior.
2. Score all candidate $l$ values by predictive variance.
3. Choose the $l$ with the highest score.
4. Run the trial against a synthetic participant.
5. Update the posterior.
6. Repeat.

In [ ]:
# ── PyMC model factory ────────────────────────────────────────────────────────
# Returns a fresh PyMC model each call, conditioning on whatever (l, y) data
# has been collected so far.  We call this inside the adaptive loop.
def make_wm_model(ls_observed, ys_observed):
    with pm.Model() as m:
        theta   = pm.Normal("theta", mu=prior_mean, sigma=prior_sd)
        logit_p = sensitivity * (theta - np.asarray(ls_observed))
        p       = pm.Deterministic("p", pm.math.sigmoid(logit_p))
        pm.Bernoulli("y", p=p, observed=np.asarray(ys_observed))
    return m

# ── Acquisition function ─────────────────────────────────────────────────────
# Scores each candidate l by posterior predictive variance of y.
# High score → the model is uncertain → trial will be informative.
def acquisition_variance(idata, candidate_ls):
    # Flatten posterior samples across chains and draws
    theta_samples = idata.posterior["theta"].values.reshape(-1)
    scores = []
    for l in candidate_ls:
        # Success probability for each sampled θ
        p_samples = 1.0 / (1.0 + np.exp(-sensitivity * (theta_samples - l)))
        # Predictive variance = E[p(1−p)] + Var[p]
        #   First term: expected Bernoulli variance at each θ
        #   Second term: additional variance due to uncertainty in θ itself
        pred_var = np.mean(p_samples * (1.0 - p_samples)) + np.var(p_samples)
        scores.append(pred_var)
    return np.array(scores)

# ── Synthetic participant ─────────────────────────────────────────────────────
# A fake subject who always succeeds when l < θ_true and always fails when l ≥ θ_true.
# Using a deterministic threshold keeps the demo clean; the model doesn't know it's deterministic.
TRUE_THETA = 6.0

def synthetic_participant(l):
    return int(l < TRUE_THETA)   # 1 if correct, 0 if not

print(f"Synthetic participant's true capacity: θ = {TRUE_THETA}")
print("Candidate sequence lengths:", np.arange(1, 15))

In [ ]:
N_ROUNDS      = 10
candidate_ls  = np.arange(1.0, 15.0)   # possible sequence lengths to test

ls_collected  = []   # l values chosen across rounds
ys_collected  = []   # corresponding outcomes
history       = []   # (posterior_mean, posterior_sd) after each round

# Initialise: represent the prior as a large sample from N(prior_mean, prior_sd).
# az.from_dict lets us treat the prior as an InferenceData object so we can
# call acquisition_variance() before any real data is collected.
idata_adaptive = az.from_dict(
    posterior={"theta": rng.normal(prior_mean, prior_sd, size=(1, 2000))}
)
history.append((prior_mean, prior_sd))   # round 0 = prior

for t in range(N_ROUNDS):
    # ── Step 1: score all candidate ls ──────────────────────────────────────
    scores = acquisition_variance(idata_adaptive, candidate_ls)
    best_l = candidate_ls[np.argmax(scores)]   # choose the most informative l

    # ── Step 2: run the trial ────────────────────────────────────────────────
    y = synthetic_participant(best_l)

    ls_collected.append(best_l)
    ys_collected.append(y)

    # ── Step 3: update the posterior with all data collected so far ──────────
    with make_wm_model(ls_collected, ys_collected):
        idata_adaptive = pm.sample(
            draws=1000, tune=1000, chains=2,
            target_accept=0.9, random_seed=42, progressbar=False
        )

    mu = idata_adaptive.posterior["theta"].values.mean()
    sd = idata_adaptive.posterior["theta"].values.std(ddof=1)
    history.append((mu, sd))

    print(f"Round {t+1:2d} | chose l={best_l:.0f} | outcome={'✓ correct' if y else '✗ wrong'} "
          f"| posterior θ ≈ {mu:.2f} ± {sd:.2f}")

print(f"\nTrue θ = {TRUE_THETA}   |   Final estimate: {history[-1][0]:.2f} ± {history[-1][1]:.2f}")

In [ ]:
# ── Visualise how the posterior narrows round by round ────────────────────────
x    = np.linspace(0, 14, 300)
cmap = plt.colormaps["winter"].resampled(len(history))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: posterior evolution
ax = axes[0]
for idx, (mu, sd) in enumerate(history):
    label = "Prior" if idx == 0 else (f"Round {idx}" if idx == len(history)-1 else None)
    ax.plot(x, norm.pdf(x, mu, sd), color=cmap(idx), lw=2, alpha=0.85, label=label)
ax.axvline(TRUE_THETA, color="black", ls="--", lw=1.5, label=f"True θ = {TRUE_THETA}")
ax.set_xlabel(r"$	heta$")
ax.set_ylabel("Density")
ax.set_title("Posterior narrows as the adaptive loop collects data")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Right: which l values were chosen?
ax = axes[1]
ax.plot(range(1, len(ls_collected)+1), ls_collected, "o-", lw=2, color="C0", label="Chosen l")
ax.axhline(TRUE_THETA, color="black", ls="--", lw=1.5, label=f"True θ = {TRUE_THETA}")
ax.set_xlabel("Round")
ax.set_ylabel("Chosen sequence length $l$")
ax.set_title("Adaptive design homes in on the threshold")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle("BOED adaptive experiment — synthetic participant", fontsize=12)
plt.tight_layout()
plt.show()

## §4  Naive vs adaptive design

The adaptive strategy chose which $l$ to test based on what would be most informative.
A naive alternative is to test $l = 1, 2, 3, \ldots, 10$ in order, regardless of outcomes.

Both strategies use the same number of trials (10) and the same model.
The question is: **does intelligent choice of $l$ lead to a sharper posterior?**

In [ ]:
# ── Naive sequential design: test l = 1, 2, ..., 10 in fixed order ────────────
naive_ls = np.arange(1.0, 11.0)
naive_ys = np.array([synthetic_participant(l) for l in naive_ls])

with make_wm_model(naive_ls, naive_ys):
    idata_naive = pm.sample(
        draws=1000, tune=1000, chains=2,
        target_accept=0.9, random_seed=123, progressbar=False
    )

# Extract posterior summaries for both strategies
mu_naive  = idata_naive.posterior["theta"].values.mean()
sd_naive  = idata_naive.posterior["theta"].values.std(ddof=1)
mu_adapt, sd_adapt = history[-1]   # final adaptive posterior (from the loop above)

print(f"After {N_ROUNDS} trials each:")
print(f"  Naive design:    θ ≈ {mu_naive:.2f} ± {sd_naive:.2f}  (posterior SD = {sd_naive:.2f})")
print(f"  Adaptive design: θ ≈ {mu_adapt:.2f} ± {sd_adapt:.2f}  (posterior SD = {sd_adapt:.2f})")
print(f"  True θ = {TRUE_THETA}")
print()
print(f"Posterior variance ratio (naive / adaptive): {sd_naive**2 / sd_adapt**2:.2f}x")
print("  > 1 means adaptive produced a more concentrated posterior")

# ── Compare posteriors ────────────────────────────────────────────────────────
x = np.linspace(0, 14, 300)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, norm.pdf(x, prior_mean, prior_sd),  "gray",    lw=1.5, ls="--",
        label=f"Prior N({prior_mean}, {prior_sd**2:.0f})", alpha=0.7)
ax.plot(x, norm.pdf(x, mu_naive,  sd_naive),   "C1",      lw=2.5,
        label=f"Naive     (σ = {sd_naive:.2f})")
ax.plot(x, norm.pdf(x, mu_adapt,  sd_adapt),   "C0",      lw=2.5,
        label=f"Adaptive  (σ = {sd_adapt:.2f})")
ax.axvline(TRUE_THETA, color="black", ls="--", lw=1.5, label=f"True θ = {TRUE_THETA}")
ax.set_xlabel(r"$\theta$ (working-memory capacity)")
ax.set_ylabel("Density")
ax.set_title(f"After {N_ROUNDS} trials: adaptive design produces a sharper posterior")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## §5  Test yourself — run the experiment on your own working memory

So far we've used a synthetic participant.  Now **you** are the participant.

At each round:
1. The model picks the sequence length $l$ it expects to be most informative.
2. A random $l$-digit number is shown for 5 seconds, then masked.
3. You type it back.
4. The model updates its posterior and picks the next $l$.

After ~8–10 rounds, the model will have a reasonable estimate of your personal working-memory
capacity.

> **Notes:**
> - The maximum sequence length is capped at 14.
> - This is a fun demo, not a rigorous cognitive test.

In [ ]:
import time
from IPython.display import display, update_display, HTML

# ── Number generator ──────────────────────────────────────────────────────────
def generate_number(length):
    """Return a random integer with exactly `length` digits."""
    if length <= 0:
        raise ValueError("length must be > 0")
    lo = 10 ** (length - 1)
    hi = 10 ** length - 1
    return int(rng.integers(lo, hi + 1))

# ── Show number, then mask after display_time seconds ─────────────────────────
def show_and_mask(number, display_time=5):
    disp = display(HTML(f"<h2>Remember this number: <code>{number}</code></h2>"),
                   display_id=True)
    time.sleep(display_time)
    # Replace the display content with asterisks
    update_display(
        HTML(f"<h2>Remember this number: <code>{'*' * len(str(number))}</code></h2>"),
        display_id=disp.display_id
    )

# ── One trial: show number, ask for recall ─────────────────────────────────────
def run_trial(length, display_time=5):
    num   = generate_number(length)
    show_and_mask(num, display_time)
    guess = input("Type the number you saw: ").strip()
    correct = int(guess == str(num))
    return correct, num

print("Setup complete. Run the cell below when you're ready to start.")

In [ ]:
# ── Interactive adaptive loop ─────────────────────────────────────────────────
# Runs the real experiment — you are the participant.

N_ROUNDS_INTERACTIVE = 10

ls_you     = []
ys_you     = []
history_you = []

# Initialise posterior as the prior
idata_you = az.from_dict(
    posterior={"theta": rng.normal(prior_mean, prior_sd, size=(1, 2000))}
)
history_you.append((prior_mean, prior_sd))

for t in range(N_ROUNDS_INTERACTIVE):
    # Pick the most informative sequence length
    scores = acquisition_variance(idata_you, candidate_ls)
    best_l = int(candidate_ls[np.argmax(scores)])
    print(f"\n--- Round {t+1} ---")
    print(f"Sequence length: {best_l}")

    # Run the actual trial
    correct, shown_number = run_trial(best_l, display_time=5)
    print(f"The number was: {shown_number}")
    print("✅ Correct!" if correct else "❌ Incorrect")

    ls_you.append(float(best_l))
    ys_you.append(correct)

    # Re-fit the model with all data collected so far
    with make_wm_model(ls_you, ys_you):
        idata_you = pm.sample(
            draws=1000, tune=1000, chains=2,
            target_accept=0.9, random_seed=42, progressbar=False
        )

    mu = idata_you.posterior["theta"].values.mean()
    sd = idata_you.posterior["theta"].values.std(ddof=1)
    history_you.append((mu, sd))
    print(f"Current estimate: θ ≈ {mu:.2f} ± {sd:.2f}")

print(f"\nFinal estimate of your working-memory capacity: θ ≈ {history_you[-1][0]:.1f} ± {history_you[-1][1]:.1f} digits")

In [ ]:
# ── Visualise your posterior evolution ────────────────────────────────────────
x    = np.linspace(0, 14, 300)
cmap = plt.colormaps["plasma"].resampled(len(history_you))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for idx, (mu, sd) in enumerate(history_you):
    label = "Prior" if idx == 0 else (f"Round {idx}" if idx == len(history_you)-1 else None)
    ax.plot(x, norm.pdf(x, mu, sd), color=cmap(idx), lw=2, alpha=0.85, label=label)
ax.set_xlabel(r"$	heta$")
ax.set_ylabel("Density")
ax.set_title("Your posterior evolution")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(range(1, len(ls_you)+1), ls_you, "o-", lw=2, color="C3")
ax.set_xlabel("Round")
ax.set_ylabel("Chosen sequence length $l$")
ax.set_title("Sequence lengths chosen by the adaptive model")
ax.grid(alpha=0.3)

plt.suptitle("Your working-memory experiment", fontsize=12)
plt.tight_layout()
plt.show()

## Key takeaways

1. **A model tells you where to look next.** The posterior predictive variance identifies
   sequence lengths where the model is still uncertain — those are the most informative trials.

2. **Adaptive design outperforms naive design.** The same number of trials produces a sharper
   posterior when sequence lengths are chosen intelligently rather than in fixed order.

3. **Predictive variance is a tractable proxy for EIG.** The true criterion (expected
   information gain, EIG = expected KL divergence) is hard to compute. Predictive variance
   is cheaper and works well in practice for one-dimensional experiments like this.

4. **The ideas scale.** The same adaptive loop applies to any experiment where you have a
   model with latent parameters, a choice of design, and binary (or continuous) outcomes:
   - Which perturbation to make in a CRISPR screen?
   - Which sequence to synthesise next in protein engineering?
   - Which patient to enrol in an adaptive clinical trial?

5. **BOED and active learning are basically the same thing.** These terms get used differently by different communities. I think it's fair to say they are similar in spirit.

---

**Coming up in Session 3b:** BOED is *sequential* — one experiment at a time, updating after each.  
Sometimes you must commit to many experiments at once (a whole plate, a synthesis batch).  
Session 3b introduces **compressed sensing**: a framework for designing those *batch* experiments
so that a sparse signal can be recovered from far fewer measurements than unknowns.  
The Horseshoe prior from Session 1 reappears as the Bayesian workhorse for that recovery.

---

## Exercises

**Q1 (easy):** Change `TRUE_THETA` to 10 and re-run the adaptive loop.  
Does the model still converge to the right value?  
How many rounds does it take to get the posterior SD below 1?

**Q2 (medium):** Replace the deterministic `synthetic_participant` with a stochastic version:
```python
def stochastic_participant(l):
    p = 1.0 / (1.0 + np.exp(-(TRUE_THETA - l)))
    return int(rng.random() < p)
```
How does this change the convergence behaviour? Does the adaptive design still outperform naive?

---

## Answer 1 — Higher threshold: TRUE\_THETA = 10

With `TRUE_THETA = 10` (at the prior mean), the model starts with a prior already centred on
the truth.  The adaptive loop still homes in correctly, but the early rounds may probe around
$l = 7$ first (where predictive variance is highest under the prior) before converging on $l = 10$.

Compare the posterior standard deviation after each round to the original `TRUE_THETA = 6` run.


In [ ]:
# ── Re-run adaptive loop with a higher true threshold ─────────────────────────
# All helper functions (make_wm_model, acquisition_variance, candidate_ls, etc.)
# are already defined in the cells above — we just change the true value.

TRUE_THETA_A1  = 10.0   # ← changed from 6 to 10
N_ROUNDS_A1    = 10

ls_a1, ys_a1, history_a1 = [], [], []
rng_a1   = np.random.default_rng(1)   # different seed so random draws differ

# Represent the prior as a large sample — same as the main loop above
idata_a1 = az.from_dict(
    posterior={"theta": rng_a1.normal(prior_mean, prior_sd, size=(1, 2000))}
)
history_a1.append((prior_mean, prior_sd))

for t in range(N_ROUNDS_A1):
    scores  = acquisition_variance(idata_a1, candidate_ls)
    best_l  = candidate_ls[np.argmax(scores)]
    y       = int(best_l < TRUE_THETA_A1)       # deterministic participant, threshold 10
    ls_a1.append(best_l);  ys_a1.append(y)

    with make_wm_model(ls_a1, ys_a1):
        idata_a1 = pm.sample(1000, tune=1000, chains=2, target_accept=0.9,
                              random_seed=42, progressbar=False)
    mu = idata_a1.posterior["theta"].values.mean()
    sd = idata_a1.posterior["theta"].values.std(ddof=1)
    history_a1.append((mu, sd))
    print(f"Round {t+1:2d} | l={best_l:.0f} | {'✓' if y else '✗'} | θ ≈ {mu:.2f} ± {sd:.2f}")

print(f"\nFinal: θ ≈ {history_a1[-1][0]:.2f} ± {history_a1[-1][1]:.2f}  (true = {TRUE_THETA_A1})")

# Compare posterior SD trajectory: original (TRUE_THETA=6) vs new (TRUE_THETA=10)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([h[1] for h in history],    'o-', color='C0', label='TRUE_THETA = 6  (original)')
ax.plot([h[1] for h in history_a1], 's-', color='C1', label='TRUE_THETA = 10')
ax.axhline(prior_sd, color='gray', ls='--', lw=1, label=f'Prior SD = {prior_sd}')
ax.set_xlabel('Round');  ax.set_ylabel('Posterior SD')
ax.set_title('Convergence comparison: θ=6 vs θ=10')
ax.legend(fontsize=9);  ax.grid(alpha=0.3)
plt.tight_layout();  plt.show()


---

## Answer 2 — Stochastic participant

The deterministic synthetic participant always succeeds when $l < \theta$ and always fails
when $l \geq \theta$.  A real person is stochastic: they succeed with probability
$\sigma(\theta - l)$, which is only 50% at $l = \theta$ and varies smoothly around it.

With a stochastic participant:
- Individual outcomes are noisy — the same $l$ might give success one round, failure the next.
- The posterior converges more slowly, but the adaptive strategy still outperforms naive.
- The model is actually *correct* about the generative process (since we built it that way).


In [ ]:
# ── Stochastic participant ────────────────────────────────────────────────────
def stochastic_participant(l):
    """Succeed with probability σ(TRUE_THETA − l); fail otherwise."""
    p = 1.0 / (1.0 + np.exp(-(TRUE_THETA - l)))
    return int(rng.random() < p)

# ── Adaptive loop with stochastic participant ─────────────────────────────────
N_ROUNDS_A2 = 10
ls_a2, ys_a2, history_a2 = [], [], []
rng_a2_state = np.random.default_rng(7)

idata_a2 = az.from_dict(
    posterior={"theta": rng_a2_state.normal(prior_mean, prior_sd, size=(1, 2000))}
)
history_a2.append((prior_mean, prior_sd))

for t in range(N_ROUNDS_A2):
    scores = acquisition_variance(idata_a2, candidate_ls)
    best_l = candidate_ls[np.argmax(scores)]
    y      = stochastic_participant(best_l)
    ls_a2.append(best_l);  ys_a2.append(y)

    with make_wm_model(ls_a2, ys_a2):
        idata_a2 = pm.sample(1000, tune=1000, chains=2, target_accept=0.9,
                              random_seed=42, progressbar=False)
    mu = idata_a2.posterior["theta"].values.mean()
    sd = idata_a2.posterior["theta"].values.std(ddof=1)
    history_a2.append((mu, sd))
    print(f"Round {t+1:2d} | l={best_l:.0f} | {'✓' if y else '✗'} | θ ≈ {mu:.2f} ± {sd:.2f}")

print(f"\nFinal: θ ≈ {history_a2[-1][0]:.2f} ± {history_a2[-1][1]:.2f}  (true = {TRUE_THETA})")
print("Notice: noisier trajectory, but posterior still converges near the true value.")
